# K-Means Robustness Decision

This notebook checks whether K-Means with `k=5` is a reasonable final cluster count for the compact RobustScaler modelling input. It does not change preprocessing or use basket data.


## Decision Logic

1. Compare K-Means results for several values of `k`.
2. Check whether the selected `k=5` result is stable across random seeds.
3. Save only the compact evidence tables needed for the project state and final report.


## Imports


In [ ]:
import sys

import matplotlib.pyplot as plt
import pandas as pd

sys.path.append("../src")

from clustering import (
    calculate_seed_stability,
    compare_kmeans_k_values,
    fit_kmeans,
    split_customer_features,
    validate_clustering_input,
)


## Load Final Modelling Input

The input is the final compact business feature set scaled with `RobustScaler`.


In [ ]:
selected_model_features = pd.read_csv("../data/processed/selected_model_features.csv")
validate_clustering_input(selected_model_features)

customer_ids, X = split_customer_features(selected_model_features)

input_check = pd.DataFrame({
    "check": [
        "rows",
        "unique customer_id",
        "model features",
        "missing values",
        "duplicated customer_id",
    ],
    "value": [
        len(selected_model_features),
        selected_model_features["customer_id"].nunique(),
        X.shape[1],
        selected_model_features.isna().sum().sum(),
        selected_model_features["customer_id"].duplicated().sum(),
    ],
})
input_check


## Compare Cluster Counts

`k=2` and `k=3` have stronger separation metrics, but they create very broad segments. `k=5` is checked as a more useful segmentation level because it gives more actionable groups while keeping acceptable balance.


In [ ]:
k_metrics = compare_kmeans_k_values(
    X,
    k_values=range(2, 11),
    random_state=42,
    n_init=50,
    sample_size=10000,
)

k_metrics = k_metrics[[
    "k",
    "random_state",
    "n_features",
    "inertia",
    "silhouette_score",
    "calinski_harabasz_score",
    "davies_bouldin_score",
    "min_cluster_size",
    "max_cluster_size",
    "min_cluster_percentage",
    "max_cluster_percentage",
]]

k_metrics.to_csv("../outputs/segmentation_robustness_metrics.csv", index=False)
k_metrics


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].plot(k_metrics["k"], k_metrics["silhouette_score"], marker="o")
axes[0].axvline(5, color="black", linestyle="--", linewidth=1)
axes[0].set_title("Silhouette")
axes[0].set_xlabel("k")

axes[1].plot(k_metrics["k"], k_metrics["davies_bouldin_score"], marker="o")
axes[1].axvline(5, color="black", linestyle="--", linewidth=1)
axes[1].set_title("Davies-Bouldin")
axes[1].set_xlabel("k")

axes[2].plot(k_metrics["k"], k_metrics["max_cluster_percentage"], marker="o", label="Largest cluster")
axes[2].plot(k_metrics["k"], k_metrics["min_cluster_percentage"], marker="o", label="Smallest cluster")
axes[2].axvline(5, color="black", linestyle="--", linewidth=1)
axes[2].set_title("Cluster Balance")
axes[2].set_xlabel("k")
axes[2].legend()

plt.tight_layout()
plt.show()


## Seed Stability For k=5

The same final feature input is refit with several random seeds. High adjusted Rand index values mean the customer assignments are not highly dependent on one random initialization.


In [ ]:
seed_stability = calculate_seed_stability(
    X,
    k=5,
    seeds=[0, 21, 42, 99, 123],
    n_init=50,
)
seed_stability.to_csv("../outputs/segmentation_seed_stability.csv", index=False)
seed_stability


## k=5 vs k=6 Business Balance Check

The main concern with `k=5` is that one cluster contains more than half of all customers. This section checks whether `k=6` creates a meaningful business split of that large `k=5` cluster, or whether it only improves balance while weakening the clustering metrics.


In [ ]:
_, k5_labels = fit_kmeans(X, n_clusters=5, random_state=42, n_init=50)
_, k6_labels = fit_kmeans(X, n_clusters=6, random_state=42, n_init=50)

k5_k6_assignments = pd.DataFrame({
    "customer_id": customer_ids,
    "k5_cluster": k5_labels,
    "k6_cluster": k6_labels,
})

largest_k5_cluster = k5_k6_assignments["k5_cluster"].value_counts().idxmax()
large_cluster_assignments = k5_k6_assignments[
    k5_k6_assignments["k5_cluster"] == largest_k5_cluster
].copy()

split_counts = (
    large_cluster_assignments.groupby(["k5_cluster", "k6_cluster"])
    .size()
    .reset_index(name="customer_count")
)
split_counts["share_of_big_k5_cluster"] = (
    split_counts["customer_count"] / len(large_cluster_assignments)
)

k6_cluster_sizes = (
    k5_k6_assignments.groupby("k6_cluster")
    .size()
    .reset_index(name="k6_cluster_total_count")
)
split_counts = split_counts.merge(k6_cluster_sizes, on="k6_cluster", how="left")
split_counts["share_of_k6_cluster_from_big_k5_cluster"] = (
    split_counts["customer_count"] / split_counts["k6_cluster_total_count"]
)
split_counts["main_split_part"] = split_counts["share_of_big_k5_cluster"] >= 0.05

split_counts.sort_values("customer_count", ascending=False)


In [ ]:
customer_features_info = pd.read_csv("../data/processed/customer_features_info.csv")

large_cluster_profile_input = large_cluster_assignments.merge(
    customer_features_info,
    on="customer_id",
    how="left",
)

profile_columns = [
    "age",
    "customer_tenure",
    "total_lifetime_spend",
    "lifetime_total_distinct_products",
    "percentage_of_products_bought_promotion",
    "number_complaints",
    "distinct_stores_visited",
    "has_loyalty_card",
    "total_children_home",
]
share_columns = [
    column for column in customer_features_info.columns
    if column.startswith("share_")
]

profile_aggregations = {
    "customer_id": "nunique",
}
for column in profile_columns:
    profile_aggregations[column] = ["mean", "median"]
for column in share_columns:
    profile_aggregations[column] = "mean"

k5_k6_split_profile = (
    large_cluster_profile_input.groupby(["k5_cluster", "k6_cluster"])
    .agg(profile_aggregations)
    .reset_index()
)
k5_k6_split_profile.columns = [
    column if statistic == "" else f"{column}_{statistic}"
    for column, statistic in k5_k6_split_profile.columns
]
k5_k6_split_profile = k5_k6_split_profile.rename(
    columns={"customer_id_nunique": "customer_count"}
)

k5_k6_split_profile = k5_k6_split_profile.merge(
    split_counts,
    on=["k5_cluster", "k6_cluster", "customer_count"],
    how="left",
)

front_columns = [
    "k5_cluster",
    "k6_cluster",
    "customer_count",
    "share_of_big_k5_cluster",
    "k6_cluster_total_count",
    "share_of_k6_cluster_from_big_k5_cluster",
    "main_split_part",
]
k5_k6_split_profile = k5_k6_split_profile[
    front_columns + [
        column for column in k5_k6_split_profile.columns
        if column not in front_columns
    ]
]

k5_k6_split_profile.to_csv(
    "../outputs/k5_k6_big_cluster_split_profile.csv",
    index=False,
)

k5_k6_split_profile


## k=5 vs k=6 Interpretation

The `k=6` solution mostly splits the large `k=5` cluster into two sizeable groups. One subgroup is older, higher-spend, higher-tenure, and more grocery-heavy. The other is younger, lower-spend, and more promotion-sensitive. This means `k=6` does reveal a readable business split.

The tradeoff is that `k=6` has weaker internal metrics than `k=5`, especially silhouette and Davies-Bouldin. The final report should therefore present `k=5` as the metric-supported final model, while noting that the largest segment contains a meaningful possible sub-split if the business prioritizes more balanced targeting.


## Recommendation


In [ ]:
k5 = k_metrics.loc[k_metrics["k"] == 5].iloc[0]
k6 = k_metrics.loc[k_metrics["k"] == 6].iloc[0]
seed_summary = seed_stability["adjusted_rand_index"].agg(["min", "mean", "max"])
main_split = k5_k6_split_profile[k5_k6_split_profile["main_split_part"]].copy()

split_summary = (
    f"k=6 splits the largest k=5 cluster into "
    f"{len(main_split)} main subgroups covering "
    f"{main_split['share_of_big_k5_cluster'].sum() * 100:.2f}% of that cluster."
)

recommendation = pd.DataFrame([
    {
        "candidate": "KMeans_k5_compact_robust",
        "evidence_for": (
            f"k=5 gives five usable segments with silhouette {k5['silhouette_score']:.3f}, "
            f"Davies-Bouldin {k5['davies_bouldin_score']:.3f}, and minimum cluster share "
            f"{k5['min_cluster_percentage']:.2f}%."
        ),
        "evidence_against": (
            f"The largest cluster is broad at {k5['max_cluster_percentage']:.2f}% of customers; "
            f"k=6 improves balance to a largest cluster of {k6['max_cluster_percentage']:.2f}% "
            f"and creates a readable split, but its silhouette falls to {k6['silhouette_score']:.3f} "
            f"and Davies-Bouldin worsens to {k6['davies_bouldin_score']:.3f}."
        ),
        "seed_stability": (
            f"ARI range {seed_summary['min']:.3f} to {seed_summary['max']:.3f}, "
            f"mean {seed_summary['mean']:.3f}."
        ),
        "business_balance_check": split_summary,
        "recommendation": "Keep K-Means k=5 as the metric-supported final segmentation level, and document the k=6 split as a business tradeoff.",
    }
])

recommendation.to_csv("../outputs/segmentation_robustness_recommendation.csv", index=False)
recommendation


## Conclusion

K-Means `k=5` is not chosen because it maximizes every internal metric. It is chosen because it balances separation, interpretability, and actionability better than the very broad lower-`k` alternatives. The largest cluster should be discussed as a broad segment in the final report.
